# RNA Velocity

Launcher na kernelu **`velo-core`** (Fig. 6D z Luo et al. 2026):

1. **Inference** — metody odpalane przez CLI w `src/rna_velocity/scripts/` (właściwy conda env per metoda; wynik: `.h5ad` z `layers['velocity']`).
2. **Downstream** — wspólny CellRank na zapisanych wynikach (lokalnie w tym notebooku).

Obce metody (DeepVelo, UniTVelo, LatentVelo, Pyro-Velocity) **nie** są importowane tutaj — tylko `subprocess` / `conda run`.

## Imports

In [1]:
import scanpy as sc

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import anndata as ad
import cellrank as cr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv

# Repo root → src on path (CLI runners use PYTHONPATH=src the same way)
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rna_velocity.registry import METHOD_ENV, METHOD_VKEY

sc.settings.verbosity = 2
scv.settings.verbosity = 2
print("methods:", list(METHOD_ENV))
print("vkeys:", METHOD_VKEY)

## Helper functions

In [ ]:
def check_velocity_adata(
    adata: ad.AnnData,
    *,
    stage: str = "inference",
    cluster_key: str = "clusters",
    require_clusters: bool = False,
    require_umap: bool = False,
    min_nonzero_frac: float = 0.0,
    raise_on_fail: bool = False,
    verbose: bool = True,
) -> dict:
    """Validate AnnData against the RNA-velocity contract.

    stage="inference"  — needs layers['spliced'] / layers['unspliced'] (CLI input)
    stage="downstream" — needs layers['velocity'] (+ neighbors for CellRank)

    Returns a report dict with ok / missing / warnings. Optionally raises.
    """
    if stage not in {"inference", "downstream"}:
        raise ValueError(f"stage must be 'inference' or 'downstream', got {stage!r}")

    missing: list[str] = []
    warnings: list[str] = []
    details: dict = {"n_obs": adata.n_obs, "n_vars": adata.n_vars, "stage": stage}

    def _layer_ok(name: str) -> bool:
        if name not in adata.layers:
            missing.append(f"layers[{name!r}]")
            return False
        layer = adata.layers[name]
        shape = getattr(layer, "shape", None)
        if shape != adata.shape:
            missing.append(f"layers[{name!r}] shape {shape} != adata.shape {adata.shape}")
            return False
        if min_nonzero_frac > 0:
            if hasattr(layer, "nnz"):
                frac = layer.nnz / (adata.n_obs * adata.n_vars)
            else:
                arr = np.asarray(layer)
                frac = float((arr != 0).mean())
            details[f"{name}_nonzero_frac"] = frac
            if frac < min_nonzero_frac:
                warnings.append(
                    f"layers[{name!r}] nonzero frac {frac:.4g} < min_nonzero_frac={min_nonzero_frac}"
                )
        return True

    if stage == "inference":
        _layer_ok("spliced")
        _layer_ok("unspliced")
    else:
        _layer_ok("velocity")
        has_neighbors = (
            "neighbors" in adata.uns
            or "connectivities" in adata.obsp
            or "distances" in adata.obsp
        )
        details["has_neighbors"] = has_neighbors
        if not has_neighbors:
            missing.append("neighbors (uns['neighbors'] or obsp connectivities/distances)")

    if require_clusters or cluster_key in adata.obs:
        if cluster_key not in adata.obs:
            missing.append(f"obs[{cluster_key!r}]")
        else:
            details["n_clusters"] = int(adata.obs[cluster_key].nunique())
    elif stage == "downstream":
        warnings.append(
            f"obs[{cluster_key!r}] absent — set require_clusters=True if CellRank needs labels"
        )

    if require_umap and "X_umap" not in adata.obsm:
        missing.append("obsm['X_umap']")
    elif "X_umap" not in adata.obsm:
        warnings.append("obsm['X_umap'] absent (needed for stream / embedding plots)")

    # hint: splicing under alternate names
    if stage == "inference" and missing:
        alt = [
            k
            for k in adata.layers
            if "splic" in k.lower() or "unsplic" in k.lower()
        ]
        if alt:
            warnings.append(f"related layer keys found (not standard names): {alt}")

    ok = len(missing) == 0
    report = {
        "ok": ok,
        "missing": missing,
        "warnings": warnings,
        "details": details,
        "layers": list(adata.layers.keys()),
    }

    if verbose:
        status = "OK" if ok else "FAIL"
        print(f"[{status}] stage={stage}  shape={adata.shape}")
        if missing:
            print("  missing:")
            for m in missing:
                print(f"    - {m}")
        if warnings:
            print("  warnings:")
            for w in warnings:
                print(f"    - {w}")
        if ok and not warnings:
            print("  all required fields present")

    if raise_on_fail and not ok:
        raise ValueError(
            f"AnnData failed velocity {stage} check: missing {missing}"
        )
    return report

### Data load

In [2]:
adata = sc.read("merged_dataset.h5ad")

In [3]:
adata.layers.keys()

KeysView(Layers with keys: )